# SiC Defect Detection: Colab runner

Runs the full pipeline on a free Colab GPU. Everything that should survive
(processed data, MLflow history, checkpoints) is kept in Google Drive under
`MyDrive/WafferMap-811/`, because Colab wipes its disk when the session ends.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

**One-time setup (Colab secrets, the key icon in the left sidebar):**
- `GITHUB_TOKEN`: needed only if the repo is private. Create a fine-grained token at
  github.com/settings/tokens with read access to `sic-defect-detection-ml`.
- `KAGGLE_USERNAME` and `KAGGLE_KEY`: only if you want Colab to download WM-811K itself
  (from kaggle.com > Settings > API > Create New Token). Not needed: your `LSWMD.pkl` is already in Drive via the My PC backup.

Run the cells top to bottom. On later sessions, just run cells 1 to 4 again;
the slow data prep is skipped because the result is already in Drive.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## 2. Connect Google Drive

In [ ]:
import os, shutil, glob
from google.colab import drive
drive.mount('/content/drive')

# Results and prepared data are saved here (created if missing)
DRIVE = '/content/drive/MyDrive/WafferMap-811'

# Where LSWMD.pkl is looked for, in order. The first is the folder that
# Google Drive for desktop backs up from your PC ("Computers > My PC" in Drive).
DATA_DIRS = [
    '/content/drive/Othercomputers/My PC/Downloads/Sofia University/ML Projects/WafferMap-811',
    DRIVE,
]
RESULTS = f'{DRIVE}/results'
PROCESSED = f'{DRIVE}/processed'
for d in (DRIVE, RESULTS, PROCESSED):
    os.makedirs(d, exist_ok=True)

REPO_DIR = '/content/sic-defect-detection-ml'

def save():
    """Copy MLflow history and run outputs to Drive. Call after each training cell."""
    for name in ('mlflow.db', 'mlartifacts', 'outputs'):
        src = f'{REPO_DIR}/{name}'
        if os.path.isdir(src):
            shutil.copytree(src, f'{RESULTS}/{name}', dirs_exist_ok=True)
        elif os.path.isfile(src):
            shutil.copy2(src, f'{RESULTS}/{name}')
    print('Saved to', RESULTS)

## 3. Get the code and install packages

In [ ]:
REPO = 'fbett5/sic-defect-detection-ml'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'

if os.path.isdir(f'{REPO_DIR}/.git'):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

# Bring back MLflow history from earlier sessions so runs accumulate
for name in ('mlflow.db', 'mlartifacts'):
    src = f'{RESULTS}/{name}'
    if os.path.isdir(src):
        shutil.copytree(src, name, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy2(src, name)

!pytest -q tests

## 4. Prepare the data

- **Already prepared in an earlier session?** It's copied from Drive (fast).
- **Otherwise** it reads `LSWMD.pkl` from your PC's backed-up WafferMap-811 folder in Drive
  (or `MyDrive/WafferMap-811/`) and, only if neither has it, downloads it from Kaggle. Takes about 5 to 10 minutes.

In [ ]:
NPZ = 'wm811k_64.npz'
os.makedirs('data/processed', exist_ok=True)

if os.path.exists(f'{PROCESSED}/{NPZ}'):
    print('Using prepared data from Drive')
    for f in glob.glob(f'{PROCESSED}/*.*'):
        shutil.copy2(f, 'data/processed/')
    !python prepare_wm811k.py --yolo-from-npz data/processed/{NPZ} --out data/processed
else:
    pkls = [f'{d}/LSWMD.pkl' for d in DATA_DIRS if os.path.exists(f'{d}/LSWMD.pkl')]
    if pkls:
        print('Copying', pkls[0], 'to local disk (2 GB, takes a few minutes)')
        os.makedirs('data/raw', exist_ok=True)
        shutil.copy2(pkls[0], 'data/raw/LSWMD.pkl')
    else:
        print('LSWMD.pkl not in Drive, downloading from Kaggle')
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        !pip install -q kaggle
        !kaggle datasets download -d qingyi/wm811k-wafer-map -p data/raw --unzip
    !python prepare_wm811k.py --pkl data/raw --out data/processed
    for f in glob.glob('data/processed/*.*'):      # npz, csv, png (not the YOLO folders)
        shutil.copy2(f, PROCESSED)
    print('Prepared data saved to Drive for next time')

from IPython.display import Image, display
display(Image('data/processed/samples.png'))
print(open('data/processed/split_summary.csv').read())

## 5. Quick check on real data (~2 min)

Training runs through `watch()`, which shows a **live dashboard** that refreshes every 10 seconds:
progress bar and GPU use, loss and validation-score curves per epoch, the confusion matrix
(which defect classes get mixed up), sample wafer maps with the model's guess (red = wrong),
and the last console lines. The stop button (■) on the cell stops training too.

In [ ]:
from sicdefect.live import watch
watch('python train_cnn.py --limit 5000 --epochs 2 --run-name quick-check --experiment quick-check')

## 6. CNN imbalance ablation

Four runs, about 15 to 25 minutes each on a T4. Free Colab sessions can disconnect
after a few hours, so `save()` runs after each one. If you get cut off, re-run
cells 1 to 4 and continue from the run that didn't finish.

In [ ]:
for imb in ['weighted_sampler', 'focal', 'class_weights', 'none']:
    watch(f'python train_cnn.py --model resnet18 --imbalance {imb} --epochs 20')
    save()

## 7. Bigger backbones

Set `BEST` to whichever imbalance method won in step 6 (highest val/test macro-F1 in step 9).

In [ ]:
BEST = 'weighted_sampler'
for model in ['efficientnet_b3', 'resnet50']:
    watch(f'python train_cnn.py --model {model} --imbalance {BEST} --pretrained --epochs 20')
    save()

## 8. YOLOv8 on the same split

In [ ]:
# The dashboard shows loss and top-1 accuracy per epoch (YOLO doesn't expose a live confusion matrix)
for m in ['yolov8n-cls.pt', 'yolov8s-cls.pt']:
    watch(f'python train_yolo.py --model {m} --epochs 30')
    save()

## 9. Compare results

To browse the full MLflow UI, download `MyDrive/WafferMap-811/results/mlflow.db`
to your PC and run `mlflow ui --backend-store-uri sqlite:///mlflow.db` in that folder.

In [ ]:
import mlflow, pandas as pd
mlflow.set_tracking_uri(f'sqlite:///{REPO_DIR}/mlflow.db')
runs = mlflow.search_runs(search_all_experiments=True)
cols = ['tags.mlflow.runName', 'metrics.test_macro_f1', 'metrics.test_balanced_acc',
        'metrics.test_defect_recall', 'metrics.best_val_macro_f1']
cols = [c for c in cols if c in runs]
table = runs[cols].dropna(subset=['metrics.test_macro_f1']).sort_values('metrics.test_macro_f1', ascending=False)
table.columns = [c.split('.')[-1] for c in table.columns]
table.round(4)

In [ ]:
# Confusion matrix of the best run
best = table.iloc[0]['runName']
for p in glob.glob(f'outputs/*{best}*/test_confusion.png'):
    display(Image(p))

## 10. Your own SEM images (dirt, scratch, nanowire, ...)

Fine-tunes an ImageNet model on your labelled SEM images. Needs cells 1 to 3 only
(not the WM-811K data).

**Put the images in Drive, one folder per class** (folder names become the class names):
```
MyDrive/SEM-artifacts/
    clean/      img001.tif ...   <- recommended: images with nothing wrong
    dirt/       ...
    scratch/    ...
    nanowire/   ...
```
TIFF (8 or 16 bit), PNG, JPG and BMP all work. Aim for 50+ images per class; 20 is the bare minimum.

`CROP` cuts the SEM info bar (kV, magnification, scale bar) off the bottom so the model
can't learn from the text. Set it to the bar's share of the image height, or 0 if your
images have no bar.

In [ ]:
SEM_DATA = '/content/drive/MyDrive/SEM-artifacts'
CROP = 0.07          # fraction of image height to cut off the bottom (info bar)
SEM_MODEL = 'resnet18'   # try efficientnet_b0 or convnext_tiny once this works

# No images yet? Uncomment to try the pipeline on fake SEM images first:
# !python tools/make_synthetic_sem.py --out data/synthetic_sem --n 60
# SEM_DATA, CROP = 'data/synthetic_sem', 0.09

from sicdefect.live import watch
watch(f'python train_sem.py --data "{SEM_DATA}" --model {SEM_MODEL} --crop-bottom {CROP} --epochs 30')
save()

# Keep the trained model in one obvious place in Drive
MODEL_DIR = f'/content/drive/MyDrive/SEM-models/sem-{SEM_MODEL}'
os.makedirs(MODEL_DIR, exist_ok=True)
for f in ('model.pt', 'model_ts.pt', 'test_report.txt', 'test_confusion.png', 'split.csv'):
    if os.path.exists(f'outputs/sem-{SEM_MODEL}/{f}'):
        shutil.copy2(f'outputs/sem-{SEM_MODEL}/{f}', MODEL_DIR)
print('Model saved to', MODEL_DIR)
display(Image(f'{MODEL_DIR}/test_confusion.png'))

### Use the model on new SEM images

Put new, unlabelled images in a Drive folder and point `NEW_IMAGES` at it. You get a CSV
with the predicted class and confidence per image, a gallery, and Grad-CAM heatmaps that
show *where* in the image the model found the artifact. Images under 70% confidence are
marked `needs_review`: check those by eye.

In [ ]:
NEW_IMAGES = '/content/drive/MyDrive/SEM-new'     # a folder (or a single image)
OUT = '/content/drive/MyDrive/SEM-new-results'

!python predict_sem.py --model {MODEL_DIR}/model.pt --input "{NEW_IMAGES}" --heatmaps --out "{OUT}"

import pandas as pd
display(pd.read_csv(f'{OUT}/predictions.csv'))
display(Image(f'{OUT}/gallery.png'))
for p in sorted(glob.glob(f'{OUT}/heatmaps/*.png'))[:6]:
    display(Image(p, width=480))

## 11. Detection track: find, box and classify defects (YOLO + CNN + engineer review)

Pipeline: **image → preprocess → YOLO detection → crop → CNN classification → engineer review → evaluation → report**,
on the public DeepPCB dataset (1,500 PCB inspection images, 6 defect types, each with a defect-free reference).
Needs cells 1 to 3 only. On a T4: data ~2 min, each YOLO run ~15 min, CNN ~5 min, evaluation ~3 min.
Results are copied to `MyDrive/WafferMap-811/results/outputs/` by `save()`.

The `refdiff` input stacks [tested image, reference, difference]: the same idea as die-to-die or
golden-image comparison in wafer inspection. The `gray` run is the ablation without a reference.

In [ ]:
# 11a. Curated dataset (+ DATASET_CARD.md, statistics, preprocessing views)
from IPython.display import Image, display
from sicdefect.live import watch
if not os.path.exists('data/raw/DeepPCB'):
    !git clone -q --depth 1 https://github.com/tangsanli5201/DeepPCB.git data/raw/DeepPCB
!python prepare_detection.py --dataset deeppcb --src data/raw/DeepPCB --out data/det/deeppcb
display(Image('data/det/deeppcb/stats_class_distribution.png'))
display(Image('data/det/deeppcb/stats_samples.png'))

In [ ]:
# 11b. YOLO detectors (configs/ holds every hyperparameter)
watch('python train_det.py --config configs/det_deeppcb_refdiff.yaml')
save()
watch('python train_det.py --config configs/det_deeppcb_gray.yaml')     # ablation: no reference image
save()

In [ ]:
# 11c. CNN second stage, trained with the detector's own false alarms as hard negatives
watch('python train_crop_cnn.py --config configs/cnn_deeppcb_refdiff.yaml '
      '--det-weights outputs/det/yolov8n-refdiff/weights/best.pt')
save()

In [ ]:
# 11d. Benchmark on the test split + automated HTML report
!python evaluate_det.py --det outputs/det/yolov8n-refdiff/weights/best.pt \
    --cnn outputs/cnn/cnn-resnet18-refdiff-hardneg/model.pt --view refdiff \
    --compare outputs/det/yolov8n-gray/weights/best.pt:gray --name deeppcb
save()
import json
m = json.load(open('outputs/eval/deeppcb/metrics.json'))
for mode in m['modes']:
    a, o = m['ap'][mode], m['ops'][mode]
    print(f"{mode:10s} mAP50 {a['mAP50']:.3f}  mAP50-95 {a['mAP50_95']:.3f}  "
          f"P {o['micro']['precision']:.3f}  R {o['micro']['recall']:.3f}  F1 {o['micro']['f1']:.3f}")
for f in ('pr_curves', 'confusion_two_stage', 'per_class'):
    display(Image(f'outputs/eval/deeppcb/figures/{f}.png'))

In [ ]:
# 11e. Engineer workflow: run the pipeline on 20 images and open the review page + reports
!python run_pipeline.py --det outputs/det/yolov8n-refdiff/weights/best.pt \
    --cnn outputs/cnn/cnn-resnet18-refdiff-hardneg/model.pt --view refdiff \
    --from-dataset data/det/deeppcb --n 20 --metrics outputs/eval/deeppcb/metrics.json --out outputs/runs/demo
save()

import subprocess
subprocess.Popen(['python', '-m', 'http.server', '8000', '--directory', f'{REPO_DIR}/outputs'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
from google.colab import output
print('Click to open (new tab):')
output.serve_kernel_port_as_window(8000, path='/runs/demo/review/index.html', anchor_text='Review page')
output.serve_kernel_port_as_window(8000, path='/eval/deeppcb/report.html', anchor_text='Benchmark report')
output.serve_kernel_port_as_window(8000, path='/runs/demo/batch_report.html', anchor_text='Batch report')

After reviewing, click **Export decisions** on the review page, upload the downloaded
`decisions_demo.json` (Files panel on the left, into the repo folder), and run the next cell. It writes
the confirmed labels as a ready YOLO dataset (the active-learning loop) and a summary of how much you had to correct.

In [ ]:
!python ingest_review.py --decisions decisions_demo.json --out outputs/runs/demo/reviewed
save()

## 12. Anomaly detection on MVTec AD (run in a fresh session)

anomalib installs its own versions of PyTorch libraries, which can clash with
the packages above. Do **Runtime > Restart session**, re-run cells 1 to 3, then run this.
MVTec AD (about 5 GB) downloads to Drive once and is reused after that.

In [ ]:
!pip install -q -r requirements-anomaly.txt
!python run_anomaly.py --model patchcore --root {DRIVE}/MVTecAD
save()
!python run_anomaly.py --model padim --root {DRIVE}/MVTecAD
save()